# Bahgat zero-shot comparison: XTTS v2 vs OmniVoice

**Goal:** one Colab session, same reference clip, same Egyptian Arabic sentence — find out which zero-shot engine sounds more like Bahgat Saber before any training.

| | XTTS v2 (Coqui) | OmniVoice (k2-fsa) |
|---|---|---|
| How it runs here | `pip install TTS`, model `tts_models/multilingual/multi-dataset/xtts_v2` | `pip install omnivoice`, model `k2-fsa/OmniVoice` (PyPI `omnivoice` 0.1.5) |
| Reference clip | 6–10 s (XTTS sweet spot ~6 s) | 3–10 s (OmniVoice sweet spot) |
| Arabic | `language="ar"` — generic Arabic only, no dialect code | `language_id="arz"` — Egyptian Arabic is a listed language (23.23 h training data); falls back to `arb` (Standard Arabic, 1,483.53 h) or auto |
| Output | 24 kHz | 24 kHz |
| Weights licence | Coqui Public Model License (CPML) — non-commercial | CC-BY-NC — non-commercial |

**Important honesty note:** this notebook tests the **upstream OmniVoice model** via its standalone pip package — the exact same model and weights (`k2-fsa/OmniVoice`) that VoiceStudio ships as its default engine. It does **not** test the VoiceStudio app itself (VoiceStudio is an Electron app + FastAPI backend; its Docker image cannot run on Colab, and the full app is not a pip package). VoiceStudio's app also applies its own mastering chain (highpass + compressor) on top of the model output, so the raw `.wav` here may sound slightly less polished than the same generation inside VoiceStudio. If OmniVoice wins here, the production offline-generation path can be either this pip package or the VoiceStudio backend on a local GPU machine — same weights.

**Prerequisites**
- Runtime → Change runtime type → **T4 GPU**. Verify §0 shows a T4 before continuing.
- Reference clip: `training/clean/xtts_reference.wav` (16 kHz mono WAV, 6–10 s, clean Bahgat speech) — produced by the dataset/clean job. If it is not on Drive yet, the §1 cell lets you upload it from your computer instead.
- Optional: the clip's transcript as `training/transcripts/xtts_reference.txt` (same basename, `.txt`). If present it is passed as `ref_text` to OmniVoice; if absent, OmniVoice auto-transcribes the clip with Whisper (documented upstream behaviour). XTTS does not use a reference transcript.
- Test sentence (identical for both engines): `يا عم انت بتقول ايه؟ انا مش فاهم حاجة من اللي انت بتقوله ده خالص`
- ~30–45 min total: two pip installs + two model downloads (~2 GB each) + two short generations. Keep the tab open.

**Outputs:** `xtts_sample.wav` and `omnivoice_sample.wav` in the session, exported to `My Drive/bahgat_voice_models/zeroshot_comparison_<date>/` in §5, plus `comparison_notes.json` (settings + durations).

## §0 — GPU check

Must show a T4 (15 GB VRAM). OmniVoice's documented VRAM floor is ~6 GB, so a T4 is comfortable; do not run this on a CPU-only runtime.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('STOP: no GPU in this runtime. Runtime -> Change runtime type -> T4 GPU, then re-run.')

## §1 — Reference clip

Looks for the clip in Drive first (`My Drive/bahgat_dataset/training/clean/xtts_reference.wav`, or the unzipped `bahgat_dataset.zip` layout), then falls back to a manual upload. Verifies sample rate / channels / duration — both engines want a short clean clip, and the file must be 16 kHz mono per the dataset contract.

In [ ]:
from google.colab import drive, files
import os, glob

drive.mount('/content/drive', force_remount=False)

REF_WAV = None
candidates = [
    '/content/drive/MyDrive/bahgat_dataset/training/clean/xtts_reference.wav',
    '/content/drive/MyDrive/training/clean/xtts_reference.wav',
    '/content/drive/MyDrive/bahgat_dataset.zip',  # handled below if it is the only thing present
]
for c in candidates:
    if c.endswith('.wav') and os.path.exists(c):
        REF_WAV = c
        break
if REF_WAV is None and os.path.exists('/content/drive/MyDrive/bahgat_dataset.zip'):
    import zipfile
    print('Found bahgat_dataset.zip on Drive — unzipping (looking for xtts_reference.wav)…')
    with zipfile.ZipFile('/content/drive/MyDrive/bahgat_dataset.zip') as z:
        z.extractall('/content/bahgat_dataset')
    hits = glob.glob('/content/bahgat_dataset/**/xtts_reference.wav', recursive=True)
    if hits:
        REF_WAV = hits[0]
if REF_WAV is None:
    print('Reference clip not found on Drive — upload training/clean/xtts_reference.wav now:')
    uploaded = files.upload()  # pick xtts_reference.wav from your computer
    REF_WAV = os.path.join('/content', list(uploaded.keys())[0])

print('REF_WAV =', REF_WAV)

# Optional reference transcript (same basename, .txt) — used as ref_text for OmniVoice only
REF_TEXT = ''
txt_candidates = [
    REF_WAV.replace('clean/xtts_reference.wav', 'transcripts/xtts_reference.txt'),
    '/content/drive/MyDrive/bahgat_dataset/training/transcripts/xtts_reference.txt',
    '/content/drive/MyDrive/training/transcripts/xtts_reference.txt',
    REF_WAV.rsplit('.', 1)[0] + '.txt',
]
for t in txt_candidates:
    if t.endswith('.txt') and os.path.exists(t):  # never open the WAV itself as text
        REF_TEXT = open(t, encoding='utf-8').read().strip()
        print('REF_TEXT from', t, '->', REF_TEXT)
        break
if not REF_TEXT:
    print('No reference transcript found — OmniVoice will auto-transcribe the clip with Whisper (upstream default).')

# Verify the clip: 16 kHz mono, 6-10 s
import soundfile as sf
info = sf.info(REF_WAV)
print(f'samplerate={info.samplerate} Hz, channels={info.channels}, duration={info.duration:.2f} s')
assert info.samplerate == 16000, 'Expected 16 kHz mono WAV per the dataset contract'
assert info.channels == 1, 'Expected mono WAV'
assert 3 <= info.duration <= 15, 'Expected a short clip (ideally 6-10 s)'

TEST_SENTENCE = 'يا عم انت بتقول ايه؟ انا مش فاهم حاجة من اللي انت بتقوله ده خالص'
print('TEST_SENTENCE =', TEST_SENTENCE)

## §2 — XTTS v2 (Coqui TTS)

Installs the Coqui `TTS` package and runs zero-shot cloning with `language="ar"` (XTTS v2 supports Arabic, but only generic `ar` — there is no Egyptian-dialect code). Output: `xtts_sample.wav` (24 kHz).

The install is heavy (~10 min) and downloads the `coqui/XTTS-v2` weights (~2 GB) on first use. If the pip install fails on a dependency conflict, re-run the cell — Colab's preinstalled packages sometimes clash with `TTS` on the first attempt.

In [ ]:
!pip install -q TTS 2>&1 | tail -n 5

from TTS.api import TTS
import torch

xtts = TTS('tts_models/multilingual/multi-dataset/xtts_v2', gpu=torch.cuda.is_available())
print('XTTS languages include ar:', 'ar' in xtts.languages if hasattr(xtts, 'languages') else 'n/a')

xtts.tts_to_file(
    text=TEST_SENTENCE,
    speaker_wav=REF_WAV,
    language='ar',  # XTTS v2: generic Arabic only, no arz/dialect code
    file_path='/content/xtts_sample.wav',
)
print('Wrote /content/xtts_sample.wav')

# Free the model + VRAM before loading OmniVoice
del xtts
import gc; gc.collect()
torch.cuda.empty_cache()

## §3 — OmniVoice (k2-fsa, the engine inside VoiceStudio)

Installs the **standalone upstream** `omnivoice` pip package (PyPI 0.1.5, repo `k2-fsa/OmniVoice`) — this is the same model and weights VoiceStudio bundles as its default engine, without the Electron app or Docker (neither can run on Colab).

- `OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)` — first call downloads the weights from Hugging Face (~2.3 GB).
- `model.generate(text=..., ref_audio=..., ref_text=...)` — `ref_text` is omitted when no transcript was found, and the model auto-transcribes the clip with Whisper.
- `language_id="arz"` (Egyptian Arabic) is tried first — it is a listed OmniVoice language. The cell prints the real `generate()` signature first: if this `omnivoice` version has no `language_id` parameter, it retries without it (auto language detection). If `arz` is rejected at runtime, change it to `"arb"` (Standard Arabic) and re-run just this cell.
- `num_step=32` is the upstream default diffusion-step count (16 is faster, 32 is the quality default).
- Output is a numpy array at 24 kHz; saved as `omnivoice_sample.wav`.

In [ ]:
!pip install -q omnivoice soundfile 2>&1 | tail -n 5

import inspect
import soundfile as sf
import torch
from omnivoice import OmniVoice

model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map='cuda:0',
    dtype=torch.float16,
)
print('generate() signature:', inspect.signature(model.generate))

gen_kwargs = dict(text=TEST_SENTENCE, ref_audio=REF_WAV, num_step=32)
if REF_TEXT:
    gen_kwargs['ref_text'] = REF_TEXT  # omit when empty -> Whisper auto-transcribes the clip

# Try Egyptian Arabic (arz) first; fall back gracefully if this version lacks language_id
try:
    audio = model.generate(language_id='arz', **gen_kwargs)
    used_lang = 'arz'
except TypeError as e:
    print('language_id not accepted by this omnivoice version (%s) — retrying with auto language' % e)
    audio = model.generate(**gen_kwargs)
    used_lang = 'auto'
print('OmniVoice language used:', used_lang)

sf.write('/content/omnivoice_sample.wav', audio[0], 24000)
print('Wrote /content/omnivoice_sample.wav')

del model
import gc; gc.collect()
torch.cuda.empty_cache()

## §4 — Listening test (fill in by ear)

Listen to the reference clip first, then both samples, in this order. Score each engine 1–5 (5 = best). Judge the **voice**, not the sentence — both say the same words.

| Criterion | XTTS v2 | OmniVoice |
|---|---|---|
| Similarity to Bahgat's voice (timbre, rasp, energy) — 1–5 |  |  |
| Egyptian Arabic intelligibility (every word clear?) — 1–5 |  |  |
| Dialect accuracy (sounds Egyptian, not Fusha / not foreign-accented?) — 1–5 |  |  |
| Artefacts (robotic buzz, cut-offs, mispronounced words, weird pauses) — notes |  |  |
| Overall winner for a Bahgat voice note |  |  |

**What to listen for, specifically:** Bahgat's voice is loud, fast, raspy and aggressive — a flat or polite delivery scores low on similarity even if it is clear. XTTS is expected to be clearer on generic Arabic but flatter and more Fusha-leaning; OmniVoice is expected to keep the dialect better (`arz`) but its Egyptian quality is unverified — that is exactly what this test measures.

**Decision rule:** if one engine is clearly ahead (≥2 points on similarity + intelligibility), it becomes the zero-shot candidate for offline batch generation. If both are bad, zero-shot is not good enough and the trained RVC model (from `bahgat_rvc_gpt-sovits_training.ipynb`) stays the only voice path. If both are good, keep both and re-test on a longer, angrier sentence.

In [ ]:
from IPython.display import Audio, display
import soundfile as sf
import os

for label, path in [
    ('0 — REFERENCE (real Bahgat)', REF_WAV),
    ('1 — XTTS v2', '/content/xtts_sample.wav'),
    ('2 — OmniVoice', '/content/omnivoice_sample.wav'),
]:
    info = sf.info(path)
    print(f'{label}: {os.path.basename(path)} — {info.duration:.2f} s, {info.samplerate} Hz, {info.channels} ch')
    display(Audio(path))

print()
print('Test sentence (both engines):', TEST_SENTENCE)

## §5 — Export both WAVs to Drive

Saves `xtts_sample.wav`, `omnivoice_sample.wav`, the reference clip copy, and a `comparison_notes.json` (settings, durations, language codes, model versions) to `My Drive/bahgat_voice_models/zeroshot_comparison_<today>/`. The VM disk is wiped when the session ends — export before closing the tab.

In [ ]:
import datetime, json, os, shutil, subprocess

today = datetime.date.today().isoformat()
out_dir = f'/content/drive/MyDrive/bahgat_voice_models/zeroshot_comparison_{today}'
os.makedirs(out_dir, exist_ok=True)

for f in ['/content/xtts_sample.wav', '/content/omnivoice_sample.wav']:
    assert os.path.exists(f), f'Missing output: {f} — re-run its section above'
    shutil.copy(f, os.path.join(out_dir, os.path.basename(f)))
shutil.copy(REF_WAV, os.path.join(out_dir, 'reference_xtts_reference.wav'))

def _dur(p):
    import soundfile as sf
    return round(sf.info(p).duration, 2)

notes = {
    'date': today,
    'test_sentence': TEST_SENTENCE,
    'reference_clip': 'training/clean/xtts_reference.wav',
    'reference_duration_s': _dur(REF_WAV),
    'reference_transcript_provided': bool(REF_TEXT),
    'xtts': {'model': 'tts_models/multilingual/multi-dataset/xtts_v2', 'language': 'ar',
            'file': 'xtts_sample.wav', 'duration_s': _dur('/content/xtts_sample.wav')},
    'omnivoice': {'model': 'k2-fsa/OmniVoice', 'package': 'omnivoice (PyPI)',
                'language_tried': 'arz (Egyptian Arabic)', 'file': 'omnivoice_sample.wav',
                'duration_s': _dur('/content/omnivoice_sample.wav')},
    'scores': {
        'xtts': {'similarity_1_5': None, 'intelligibility_1_5': None, 'dialect_1_5': None, 'notes': ''},
        'omnivoice': {'similarity_1_5': None, 'intelligibility_1_5': None, 'dialect_1_5': None, 'notes': ''},
        'winner': None,
    },
}
with open(os.path.join(out_dir, 'comparison_notes.json'), 'w', encoding='utf-8') as f:
    json.dump(notes, f, ensure_ascii=False, indent=2)

print('Exported to:', out_dir)
print(subprocess.run(['ls', '-la', out_dir], capture_output=True, text=True).stdout)